In [10]:
%load_ext autoreload
%autoreload 2 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

In [12]:
output_dir = Path("output")
datasets = ["AdventureWorks", "OHR", "OurAirports", "sakila"]

results = []
for dataset in datasets:
    for model_type in ["generic", "specialised"]:
        path = output_dir / f"{model_type}-{dataset}" / "results.csv"
        if path.exists():
            df = pd.read_csv(path)
            df["dataset"] = dataset
            df["type"] = model_type
            results.append(df)

results_df = pd.concat(results, ignore_index=True)

pct_cols = ["fone", "accuracy", "precision", "recall", "fpr"]
for col in pct_cols:
    if col in results_df.columns:
        results_df[col] = results_df[col].str.rstrip("%").astype(float)

recall_cols = [c for c in results_df.columns if c.startswith("recall") and c != "recall"]
for col in recall_cols:
    if results_df[col].dtype == object:
        results_df[col] = results_df[col].str.rstrip("%").astype(float)

results_df

,model,fone,accuracy,precision,recall,fpr,auprc,rocauc,auroc_ci,auprc_ci,recalltime,recallboolean,recallstacked,recallerror,recallunion,recallinsider,recallinline,dataset,type
0,Li and AE-scaler,52.45,83.06,36.52,92.98,18.04375,0.7957,0.9627,0.0004,0.0008,94.84,85.00,83.46,99.61,99.09,93.71,51.39,AdventureWorks,generic
1,Li and AE-scaler,51.57,81.19,34.78,99.68,20.87147,0.7918,0.9810,0.0003,0.0008,99.64,99.79,98.26,99.96,99.80,97.20,91.67,AdventureWorks,specialised
2,Li and AE-scaler,94.98,98.93,90.70,99.67,1.15355,0.9983,0.9994,0.0000,0.0001,92.43,99.83,87.97,99.46,99.20,91.30,99.07,OHR,generic
3,Li and AE-scaler,40.18,69.79,25.14,100.00,33.62281,0.9923,0.9985,0.0001,0.0002,99.90,100.00,100.00,100.00,100.00,100.00,100.00,OHR,specialised
4,Li and AE-scaler,35.85,67.19,22.29,91.53,35.52057,0.7867,0.9120,0.0006,0.0008,72.59,95.84,78.09,86.10,95.55,91.67,80.60,OurAirports,generic
5,Li and AE-scaler,92.12,98.53,99.29,85.92,0.06857,0.9196,0.9671,0.0003,0.0005,59.98,91.59,67.05,79.60,92.69,47.62,75.12,OurAirports,specialised
6,Li and AE-scaler,74.45,93.11,59.67,98.97,7.55613,0.9907,0.9974,0.0001,0.0002,76.16,99.49,76.83,96.14,98.72,9.09,99.43,sakila,generic
7,Li and AE-scaler,57.13,84.80,40.02,99.83,16.89975,0.9947,0.9980,0.0001,0.0001,95.59,99.94,97.12,97.99,99.39,95.45,100.00,sakila,specialised


## Main Metrics Comparison

In [ ]:
# Main metrics comparison
main_metrics = ["accuracy", "precision", "recall", "fone", "rocauc", "auprc"]
metric_labels = {
    "accuracy": "Accuracy (%)",
    "precision": "Precision (%)",
    "recall": "Recall (%)",
    "fone": "F1 Score (%)",
    "rocauc": "ROC AUC",
    "auprc": "AUPRC"
}

fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=[metric_labels[m] for m in main_metrics],
    vertical_spacing=0.15
)

colors = {"generic": "#636EFA", "specialised": "#EF553B"}

for idx, metric in enumerate(main_metrics):
    row = idx // 3 + 1
    col = idx % 3 + 1
    
    for model_type in ["generic", "specialised"]:
        subset = results_df[results_df["type"] == model_type]
        fig.add_trace(
            go.Bar(
                x=subset["dataset"],
                y=subset[metric],
                name=model_type.capitalize(),
                marker_color=colors[model_type],
                showlegend=(idx == 0),
                legendgroup=model_type
            ),
            row=row, col=col
        )

fig.update_layout(
    title="Generic vs Specialised Model Performance",
    barmode="group",
    height=600,
    width=1000
)
fig.show()

## ROC Curves Comparison

In [16]:
# Load and plot ROC curves
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=datasets,
    vertical_spacing=0.12
)

for idx, dataset in enumerate(datasets):
    row = idx // 2 + 1
    col = idx % 2 + 1
    
    for model_type in ["generic", "specialised"]:
        roc_path = output_dir / f"{model_type}-{dataset}" / "roc_curves" / "ae_li.csv"
        if roc_path.exists():
            roc_df = pd.read_csv(roc_path)
            
            auc_val = results_df[(results_df["dataset"] == dataset) & (results_df["type"] == model_type)]["rocauc"].values[0]
            
            fig.add_trace(
                go.Scatter(
                    x=roc_df["fpr"],
                    y=roc_df["tpr"],
                    mode="lines",
                    name=f"{model_type.capitalize()}",
                    line=dict(color=colors[model_type]),
                    showlegend=(idx == 0),
                    legendgroup=model_type
                ),
                row=row, col=col
            )
    
    # Add diagonal
    fig.add_trace(
        go.Scatter(
            x=[0, 1], y=[0, 1],
            mode="lines",
            line=dict(dash="dash", color="gray"),
            showlegend=False
        ),
        row=row, col=col
    )
    
    fig.update_xaxes(title_text="FPR", row=row, col=col)
    fig.update_yaxes(title_text="TPR", row=row, col=col)

fig.update_layout(
    title="ROC Curves: Generic vs Specialised",
    height=700,
    width=900
)
fig.show()